In [6]:
!pip install -Uq langchain langchain_community langchain_huggingface kiwipiepy faiss-gpu-cu12 rank_bm25

In [7]:
from google.colab import drive, userdata
import os

# 1. 드라이브 마운트
drive.mount('/content/drive')

# 2. 캐시 경로 설정 (드라이브 내 원하는 경로)
cache_dir = "/content/drive/MyDrive/hf_cache"
#os.makedirs(cache_dir, exist_ok=True)

# 3. 환경 변수 설정 (Hugging Face 라이브러리가 이 경로를 참조하게 함)
os.environ['HF_HOME'] = cache_dir
#os.environ['HF_DATASETS_CACHE'] = os.path.join(cache_dir, "datasets")
os.environ['SENTENCE_TRANSFORMERS_HOME'] = os.path.join(cache_dir, "sentence_transformers")

print("✅ 모든 AI 모델 및 데이터셋 캐시 경로가 고정되었습니다.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ 모든 AI 모델 및 데이터셋 캐시 경로가 고정되었습니다.


In [3]:
import os
from langchain_huggingface import HuggingFaceEmbeddings
import pandas as pd
from tqdm import tqdm

import json
from typing import List, Dict, Set
from kiwipiepy import Kiwi
from collections import defaultdict

import pickle

import numpy as np
from collections import defaultdict
from datetime import datetime

from langchain_community.vectorstores import FAISS

class TechMatchEvaluator:
    def __init__(self, tech_dict_path: str, tech_sym_dict: dict, bm25_by_position: dict):
        self.kiwi = Kiwi()
        self.bm25_by_position = bm25_by_position

        # 1. 영어 기술 사전 로드
        with open(tech_dict_path, "r", encoding="utf-8") as f:
            self.tech_dict = json.load(f)

        # 2. 효율적인 조회를 위해 각 포지션별 id_to_idx 매핑 생성
        self.id_to_idx_map = {}
        for pos, entry in self.bm25_by_position.items():
            db_ids = entry.get('db_ids', [])
            self.id_to_idx_map[pos] = {db_id: i for i, db_id in enumerate(db_ids)}

    def _get_query_tech_profile(self, text: str, position: str) -> Set[str]:
        """쿼리 텍스트를 실시간으로 토크나이징하여 기술 스택 프로필 생성"""
        # 해당 포지션의 기술 사전 가져오기 (없으면 전체 사전 합집합)
        valid_techs = set(self.tech_dict.get(position, []))
        if not valid_techs:
            valid_techs = set(word for techs in self.tech_dict.values() for word in techs)

        # Kiwi 토크나이징 (NNG, SL 등 주요 품사 포함)
        tokens = self.kiwi.tokenize(text)

        # 소문자 변환 후 사전에 있는 단어만 필터링
        found_techs = set()

        for t in tokens:
            word = t.form.lower()

            normalized_word = tech_sym_dict.get(word, word)

            if normalized_word in valid_techs:
                found_techs.add(normalized_word)

        return found_techs

    def analyze_matches(self, query_text: str, retrieved_ids: List, position: str):
        """쿼리와 검색된 문서들 간의 기술 매칭 분석 수행"""
        # 1. 쿼리 프로필 추출
        query_techs = self._get_query_tech_profile(query_text, position)

        # 2. 분석 대상 인덱스 정보 가져오기
        entry = self.bm25_by_position.get(position)
        if not entry or not query_techs:
            return {"query_techs": list(query_techs), "results": []}

        bm25_obj = entry['bm25']
        pos_id_map = self.id_to_idx_map.get(position, {})
        valid_techs = set(self.tech_dict.get(position, []))

        analysis_results = []

        # 3. 검색된 문서들 순회
        for db_id in retrieved_ids:
            idx = pos_id_map.get(db_id)

            if idx is not None:
                # BM25 인덱스의 미리 계산된 토큰 활용 (초고속)
                doc_raw_tokens = set(bm25_obj.doc_freqs[idx].keys())
                doc_techs = doc_raw_tokens.intersection(valid_techs)

                # 매칭 분석
                matched = query_techs.intersection(doc_techs)
                missing = query_techs - doc_techs

                # 점수 계산
                match_ratio = len(matched) / len(query_techs) if query_techs else 0

                analysis_results.append({
                    "db_id": db_id,
                    "match_ratio": round(match_ratio, 4),
                    "matched_count": len(matched),
                    "matched_list": sorted(list(matched)),
                    "missing_list": sorted(list(missing))
                })

        return {
            "query_techs": sorted(list(query_techs)),
            "results": analysis_results
        }

In [ ]:
print("⏳ 임베딩 모델 로딩 중...")

hf_embeddings = HuggingFaceEmbeddings(
    model_name="Qwen/Qwen3-Embedding-0.6B",
    model_kwargs={'device': 'cuda'}, # GPU 없으면 'cpu'
    encode_kwargs={'normalize_embeddings': True},
    cache_folder = os.environ['SENTENCE_TRANSFORMERS_HOME']
)

tech_sym_dict = {
"frontend engineer":{
    "database":"db", #단어 통일
    "테스트":"testing",
    "해커톤":"hackathon",
    "모바일": "mobile",
    "인턴": "intern"
},
"backend engineer":{
    "해커톤":"hackathon",
    "데이터베이스": "db",
    "클라우드": "cloud",
    "쿼리":"query",
    "서버": "server",
    "프레임": "framework", #프레임/워크
    "메시징": "messaging",
    "컨테이너": "container"
},
"ai engineer":{
    "해커톤":"hackathon",
    "머신": "ml", #머신/러닝 분리됨
    "텍스트": "text",
    "챗봇": "chatbot",
    "벡터": "vector",
    "튜닝": "tuning",
    "데이터베이스": "db",
    "클라우드": "cloud",
    "모델": "model",
}
}

index_path = "/content/drive/MyDrive/mysql-faiss-retriever-playground/bm25_index.pkl"
with open(index_path, 'rb') as f:
    bm25_by_position = pickle.load(f)

vectorstore = FAISS.load_local(
        folder_path = "/content/drive/MyDrive/mysql-faiss-retriever-playground/data/faiss_index_high",
        embeddings = hf_embeddings,
        allow_dangerous_deserialization=True
)

evaluator = TechMatchEvaluator("/content/drive/MyDrive/mysql-faiss-retriever-playground/tech_keyword_dict.json", tech_sym_dict, bm25_by_position)

test_data_path = "/content/drive/MyDrive/mysql-faiss-retriever-playground/evaluation_df_k5.json"
test_df = pd.read_json(test_data_path, orient = "records", encoding = "utf-8")


# 1. 통계를 위한 저장소 초기화
rank_stats = defaultdict(lambda: {1: [], 2: [], 3: []})
detailed_reports = []

test_limit = len(test_df)  # 테스트할 샘플 수
print(f"\n🔍 HybridRetriever 정밀 평가 시작 (대상: {test_limit}건)")

for idx, row in tqdm(test_df[:test_limit].iterrows(), total=test_limit):
    position = row['position_type']
    query_text = row['resume_cleaned']
    query_id = idx + 1

    # 리트리버 검색 수행
    docs = vectorstore.similarity_search(query_text, k = 5)
    final_candidate_docs = docs[:3]
    retrieved_ids = [int(d.page_content) for d in final_candidate_docs]
    
    # 매칭 분석 수행
    analysis = evaluator.analyze_matches(query_text, retrieved_ids, position)

    # 순위별 점수 기록 및 상세 데이터 수집
    current_query_report = {
        "query_id": query_id,
        "position": position,
        "query_techs": analysis['query_techs'],
        "results": []
    }

    for i, res in enumerate(analysis['results']):
        rank = i + 1
        if rank <= 3:
            # 통계용 데이터 축적
            rank_stats[position][rank].append(res['match_ratio'])
            # 상세 리포트용 데이터 축적
            current_query_report["results"].append(res)

    detailed_reports.append(current_query_report)

# 2. 통계 요약(Summary) 계산
summary_stats = {
    "by_position": {},
    "overall": {},
    "evaluated_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "sample_size": test_limit
}

all_r1, all_r2, all_r3 = [], [], []

for pos, ranks in rank_stats.items():
    r1_avg = np.mean(ranks[1]) if ranks[1] else 0
    r2_avg = np.mean(ranks[2]) if ranks[2] else 0
    r3_avg = np.mean(ranks[3]) if ranks[3] else 0

    summary_stats["by_position"][pos] = {
        "rank_1_avg": round(r1_avg, 4),
        "rank_2_avg": round(r2_avg, 4),
        "rank_3_avg": round(r3_avg, 4),
        "total_avg": round(np.mean([r1_avg, r2_avg, r3_avg]), 4)
    }

    all_r1.extend(ranks[1])
    all_r2.extend(ranks[2])
    all_r3.extend(ranks[3])

summary_stats["overall"] = {
    "rank_1_avg": round(np.mean(all_r1), 4) if all_r1 else 0,
    "rank_2_avg": round(np.mean(all_r2), 4) if all_r2 else 0,
    "rank_3_avg": round(np.mean(all_r3), 4) if all_r3 else 0,
    "total_avg": round(np.mean(all_r1 + all_r2 + all_r3), 4) if all_r1 else 0
}

# 3. 결과 출력 (Console)
print(f"\n{'='*75}")
print(f"🏆 [Final Report] 직무 및 순위별 기술 매칭 통계")
print(f"{'='*75}")
print(f"{'직무 (Position)':<25} | {'1순위':^12} | {'2순위':^12} | {'3순위':^12}")
print(f"{'-'*75}")

for pos, stats in summary_stats["by_position"].items():
    print(f"{pos:<25} | {stats['rank_1_avg']*100:>10.2f}% | {stats['rank_2_avg']*100:>10.2f}% | {stats['rank_3_avg']*100:>10.2f}%")

print(f"{'-'*75}")
o = summary_stats["overall"]
print(f"{'TOTAL AVERAGE':<25} | {o['rank_1_avg']*100:>10.2f}% | {o['rank_2_avg']*100:>10.2f}% | {o['rank_3_avg']*100:>10.2f}%")
print(f"{'='*75}\n")

# 4. JSON 파일로 저장
final_output = {
    "summary": summary_stats,
    "details": detailed_reports
}

output_filename = f"/content/drive/MyDrive/mysql-faiss-retriever-playground/evaluation_results_{datetime.now().strftime('%m%d_%H%M')}.json"
with open(output_filename, "w", encoding="utf-8") as f:
    json.dump(final_output, f, ensure_ascii=False, indent=4)

print(f"✅ 평가 결과가 '{output_filename}'에 저장되었습니다.")

⏳ 임베딩 모델 로딩 중...


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]


🔍 HybridRetriever 정밀 평가 시작 (대상: 4005건)


100%|██████████| 4005/4005 [12:04<00:00,  5.53it/s]



🏆 [Final Report] 직무 및 순위별 기술 매칭 통계
직무 (Position)             |     1순위      |     2순위      |     3순위     
---------------------------------------------------------------------------
frontend engineer         |      83.27% |      82.19% |      81.94%
backend engineer          |      78.70% |      77.71% |      77.28%
ai engineer               |      73.88% |      71.53% |      71.71%
---------------------------------------------------------------------------
TOTAL AVERAGE             |      78.91% |      77.50% |      77.32%

✅ 평가 결과가 '/content/drive/MyDrive/mysql-faiss-retriever-playground/evaluation_results_0422_0613.json'에 저장되었습니다.
